## Piper TTS — Bangla (`bn_BD-google-medium`)

Official Piper voice: 16 speakers, trained on Google's Bangladeshi Bangla dataset ([OpenSLR 37](http://www.openslr.org/37/)). 77 MB ONNX model.

Piper is built for CPU, so the CPU numbers are the ones that matter. The GPU section is only for a rough comparison.
**Runtime → Change runtime type → T4 GPU** (optional), then run all cells.

RTF = generation time / audio length. Lower is faster.

In [ ]:
!pip install -q piper-tts huggingface_hub soundfile bangla bnnumerizer bnunicodenormalizer
# GPU build of onnxruntime (also contains the CPU provider). Installed last so it replaces the CPU-only build.
!pip install -q --force-reinstall --no-deps onnxruntime-gpu

In [ ]:
from huggingface_hub import hf_hub_download

VOICE = "bn_BD-google-medium"
MODEL_PATH = hf_hub_download("rhasspy/piper-voices", f"bn/bn_BD/google/medium/{VOICE}.onnx", local_dir="/content/model")
hf_hub_download("rhasspy/piper-voices", f"bn/bn_BD/google/medium/{VOICE}.onnx.json", local_dir="/content/model")
print(MODEL_PATH)

### Helpers (same as `PiperTTS/main.py`)

In [ ]:
import os, time
import numpy as np
import soundfile as sf
import onnxruntime as ort
from IPython.display import Audio, display
from bangla import convert_english_digit_to_bangla_digit
from bnnumerizer import numerize
from bnunicodenormalizer import Normalizer
from piper import PiperVoice, SynthesisConfig

print("onnxruntime providers:", ort.get_available_providers())

OUTPUT_DIR = "/content/outputs"
SENTENCE_PAUSE = 0.25
_bnorm = Normalizer()


def normalize_text(text):
    text = convert_english_digit_to_bangla_digit(text)
    text = numerize(text)
    text = text.replace("\n", " ")
    words = [_bnorm(w)["normalized"] for w in text.split()]
    return " ".join(w for w in words if w)


def load_model(use_cuda=False):
    start = time.time()
    voice = PiperVoice.load(MODEL_PATH, use_cuda=use_cuda)
    list(voice.synthesize("ধন্যবাদ।"))  # warm-up
    print(f"loaded ({'gpu' if use_cuda else 'cpu'}) in {time.time() - start:.1f}s")
    return voice


def show_phonemes(voice, text):
    for sentence in voice.phonemize(normalize_text(text)):
        print("".join(sentence))


def generate(voice, text, filename, speaker_id=0, speed=1.0, play=True):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    sr = voice.config.sample_rate
    silence = np.zeros(int(SENTENCE_PAUSE * sr), dtype=np.float32)
    config = SynthesisConfig(speaker_id=speaker_id, length_scale=1.0 / speed)
    start = time.time()
    chunks = []
    for chunk in voice.synthesize(normalize_text(text), syn_config=config):
        chunks += [chunk.audio_float_array, silence]
    audio = np.concatenate(chunks[:-1])
    elapsed = time.time() - start
    duration = len(audio) / sr
    path = f"{OUTPUT_DIR}/{filename}.wav"
    sf.write(path, audio, sr)
    print(f"{filename}: audio {duration:.1f}s | generated in {elapsed:.3f}s | RTF {elapsed / duration:.3f}")
    if play:
        display(Audio(path))
    return {"file": filename, "audio_s": round(duration, 2), "gen_s": round(elapsed, 3), "rtf": round(elapsed / duration, 3)}

### 1. Listen to all 16 speakers and pick the best one

In [ ]:
voice = load_model(use_cuda=False)
for sid in range(voice.config.num_speakers):
    print(f"--- speaker {sid}")
    generate(voice, "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি। আজকের আবহাওয়া খুব সুন্দর।", f"speaker_{sid:02d}", sid)

### 2. Test sentences with your chosen speaker

In [ ]:
SPEAKER = 0  # change after listening above

TESTS = {
    "word": "ধন্যবাদ",
    "sentence1": "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "sentence2": "আজকের আবহাওয়া খুব সুন্দর, চলো আমরা বাইরে ঘুরতে যাই।",
    "numbers": "আমার বয়স 25 বছর, আর আজ ২৪ তারিখ।",
    "question": "তুমি কেমন আছো? আমি ভালো আছি!",
    "paragraph": (
        "বাংলাদেশ দক্ষিণ এশিয়ার একটি সুন্দর দেশ। এই দেশের মানুষ খুবই অতিথিপরায়ণ। "
        "এখানে ছয়টি ঋতু আছে, এবং প্রতিটি ঋতুর নিজস্ব সৌন্দর্য রয়েছে। "
        "নদীমাতৃক এই দেশে পদ্মা, মেঘনা ও যমুনা প্রধান নদী। "
        "ঢাকা বাংলাদেশের রাজধানী এবং সবচেয়ে বড় শহর।"
    ),
}

# What espeak will pronounce. Watch for an extra 'ɔ' inside words, e.g. আমরা -> 'amɔra' instead of 'amra'.
show_phonemes(voice, TESTS["sentence2"])

cpu_results = [{"device": "cpu", **generate(voice, t, f"cpu_{k}", SPEAKER)} for k, t in TESTS.items()]

### 3. GPU (rough comparison only)

In [ ]:
gpu_results = []
if "CUDAExecutionProvider" in ort.get_available_providers():
    gpu_voice = load_model(use_cuda=True)
    gpu_results = [{"device": "gpu", **generate(gpu_voice, t, f"gpu_{k}", SPEAKER, play=False)} for k, t in TESTS.items()]
else:
    print("No CUDA provider: switch the runtime to T4 GPU and re-run from the top.")

### Summary

In [ ]:
import pandas as pd
df = pd.DataFrame(cpu_results + gpu_results)
display(df)
display(df.groupby("device")[["audio_s", "gen_s"]].sum().assign(rtf=lambda d: (d.gen_s / d.audio_s).round(3)))

### Try your own text

In [ ]:
generate(voice, "এখানে আপনার নিজের বাংলা লেখা দিন।", "custom", speaker_id=SPEAKER, speed=1.0)